# Smart Health Insurance Predictor — Training Notebook


In [88]:
import pandas as pd, numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import joblib, json

In [89]:
# Load your dataset
csv_path = Path('premiums.csv')  # put the CSV next to this notebook
df = pd.read_csv(csv_path)
df.head()

,Age,Gender,Region,Marital_status,Number Of Dependants,BMI_Category,Smoking_Status,Employment_Status,Income_Level,Income_Lakhs,Medical History,Insurance_Plan,Annual_Premium_Amount
0,26,Male,Northwest,Unmarried,0,Normal,No Smoking,Salaried,<10L,6,Diabetes,Bronze,9053
1,29,Female,Southeast,Married,2,Obesity,Regular,Salaried,<10L,6,Diabetes,Bronze,16339
2,49,Female,Northeast,Married,2,Normal,No Smoking,Self-Employed,10L - 25L,20,High blood pressure,Silver,18164
3,30,Female,Southeast,Married,3,Normal,No Smoking,Salaried,> 40L,77,No Disease,Gold,20303
4,18,Male,Northeast,Unmarried,0,Overweight,Regular,Self-Employed,> 40L,99,High blood pressure,Silver,13365


In [90]:
# Shape of data-set
df.shape

(50000, 13)

In [91]:
# Numerical column names

df.select_dtypes(include=['number']).columns

Index(['Age', 'Number Of Dependants', 'Income_Lakhs', 'Annual_Premium_Amount'], dtype='object')

In [92]:
# Categorical column names
df.select_dtypes(include=['object']).columns

Index(['Gender', 'Region', 'Marital_status', 'BMI_Category', 'Smoking_Status',
       'Employment_Status', 'Income_Level', 'Medical History',
       'Insurance_Plan'],
      dtype='object')

In [93]:
# DataTypes of columns
df.dtypes

Age                       int64
Gender                   object
Region                   object
Marital_status           object
Number Of Dependants      int64
BMI_Category             object
Smoking_Status           object
Employment_Status        object
Income_Level             object
Income_Lakhs              int64
Medical History          object
Insurance_Plan           object
Annual_Premium_Amount     int64
dtype: object

In [94]:
# Dataset information
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Age                    50000 non-null  int64 
 1   Gender                 50000 non-null  object
 2   Region                 50000 non-null  object
 3   Marital_status         50000 non-null  object
 4   Number Of Dependants   50000 non-null  int64 
 5   BMI_Category           50000 non-null  object
 6   Smoking_Status         49989 non-null  object
 7   Employment_Status      49998 non-null  object
 8   Income_Level           49987 non-null  object
 9   Income_Lakhs           50000 non-null  int64 
 10  Medical History        50000 non-null  object
 11  Insurance_Plan         50000 non-null  object
 12  Annual_Premium_Amount  50000 non-null  int64 
dtypes: int64(4), object(9)
memory usage: 5.0+ MB


In [95]:
# Finding missing values
missing_values = df.isnull().sum() 
missing_values 

Age                       0
Gender                    0
Region                    0
Marital_status            0
Number Of Dependants      0
BMI_Category              0
Smoking_Status           11
Employment_Status         2
Income_Level             13
Income_Lakhs              0
Medical History           0
Insurance_Plan            0
Annual_Premium_Amount     0
dtype: int64

In [96]:
# Finding duplicated rows
df.duplicated().sum()

np.int64(0)

In [97]:
# Finding Unique values count for each column
df.nunique()

Age                         60
Gender                       2
Region                       4
Marital_status               2
Number Of Dependants         8
BMI_Category                 4
Smoking_Status               6
Employment_Status            3
Income_Level                 4
Income_Lakhs               110
Medical History              9
Insurance_Plan               3
Annual_Premium_Amount    23183
dtype: int64

In [98]:
# Printing unique values in each column
for col in df.select_dtypes(include=['object']).columns:
    print(f"Unique values in {col}: {df[col].unique()}")

Unique values in Gender: ['Male' 'Female']
Unique values in Region: ['Northwest' 'Southeast' 'Northeast' 'Southwest']
Unique values in Marital_status: ['Unmarried' 'Married']
Unique values in BMI_Category: ['Normal' 'Obesity' 'Overweight' 'Underweight']
Unique values in Smoking_Status: ['No Smoking' 'Regular' 'Occasional' nan 'Smoking=0' 'Does Not Smoke'
 'Not Smoking']
Unique values in Employment_Status: ['Salaried' 'Self-Employed' 'Freelancer' nan]
Unique values in Income_Level: ['<10L' '10L - 25L' '> 40L' '25L - 40L' nan]
Unique values in Medical History: ['Diabetes' 'High blood pressure' 'No Disease'
 'Diabetes & High blood pressure' 'Thyroid' 'Heart disease'
 'High blood pressure & Heart disease' 'Diabetes & Thyroid'
 'Diabetes & Heart disease']
Unique values in Insurance_Plan: ['Bronze' 'Silver' 'Gold']


In [99]:
# Summary statistics for numerical columns 
df.describe()

,Age,Number Of Dependants,Income_Lakhs,Annual_Premium_Amount
count,50000.000000,50000.000000,50000.000000,50000.000000
mean,34.593480,1.712080,23.018200,15768.116320
std,15.000437,1.498248,24.219197,8419.839675
min,18.000000,-3.000000,1.000000,3501.000000
25%,22.000000,0.000000,7.000000,8608.000000
50%,31.000000,2.000000,17.000000,13929.000000
75%,45.000000,3.000000,31.000000,22275.250000
max,356.000000,5.000000,930.000000,43471.000000


In [100]:
# Summary statistics for categorical columns 
df.describe(include=['object'])

,Gender,Region,Marital_status,BMI_Category,Smoking_Status,Employment_Status,Income_Level,Medical History,Insurance_Plan
count,50000,50000,50000,50000,49989,49998,49987,50000,50000
unique,2,4,2,4,6,3,4,9,3
top,Male,Southeast,Unmarried,Normal,No Smoking,Salaried,<10L,No Disease,Bronze
freq,27480,17520,25681,23511,27366,20968,18667,21177,21573


In [101]:
# Imbalance check and value count for 'gender' column 
gender_counts = df['Gender'].value_counts()

print("value count of",gender_counts)
print("\nClass Distribution (Imbalance Check for Gender):")
print(df['Gender'].value_counts(normalize=True) * 100)

value count of Gender
Male      27480
Female    22520
Name: count, dtype: int64

Class Distribution (Imbalance Check for Gender):
Gender
Male      54.96
Female    45.04
Name: proportion, dtype: float64


In [102]:
# imbalance check and value count for 'smoking_status' column 

smoking_status_counts = df['Smoking_Status'].value_counts()

print("value count of ",smoking_status_counts)
print("\nClass Distribution (Imbalance Check for Smoking_Status):")
print(df['Smoking_Status'].value_counts(normalize=True) * 100)


value count of  Smoking_Status
No Smoking        27366
Regular           15686
Occasional         6915
Smoking=0             8
Not Smoking           8
Does Not Smoke        6
Name: count, dtype: int64

Class Distribution (Imbalance Check for Smoking_Status):
Smoking_Status
No Smoking        54.744044
Regular           31.378903
Occasional        13.833043
Smoking=0          0.016004
Not Smoking        0.016004
Does Not Smoke     0.012003
Name: proportion, dtype: float64


## 🔍 Check Missing Values


In [103]:
print('Missing values before handling:')
print(df.isnull().sum())

# Fill missing values
for col in df.columns:
    if df[col].dtype in [np.float64, np.int64]:
        df[col].fillna(df[col].median(), inplace=True)
    else:
        df[col].fillna(df[col].mode()[0], inplace=True)

print('\nMissing values after handling:')
print(df.isnull().sum())

Missing values before handling:
Age                       0
Gender                    0
Region                    0
Marital_status            0
Number Of Dependants      0
BMI_Category              0
Smoking_Status           11
Employment_Status         2
Income_Level             13
Income_Lakhs              0
Medical History           0
Insurance_Plan            0
Annual_Premium_Amount     0
dtype: int64

Missing values after handling:
Age                      0
Gender                   0
Region                   0
Marital_status           0
Number Of Dependants     0
BMI_Category             0
Smoking_Status           0
Employment_Status        0
Income_Level             0
Income_Lakhs             0
Medical History          0
Insurance_Plan           0
Annual_Premium_Amount    0
dtype: int64


C:\Users\rohit\AppData\Local\Temp\ipykernel_14152\3866709791.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
C:\Users\rohit\AppData\Local\Temp\ipykernel_14152\3866709791.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For examp

## 📊 Outlier Detection & Handling (IQR Method)

In [ ]:
# IQR method (Interquartile Range)


def cap_outliers(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return np.clip(series, lower_bound, upper_bound)

num_cols = df.select_dtypes(include=[np.number]).columns

print('Fixing outliers in numeric columns...')
for col in num_cols:
    df[col] = cap_outliers(df[col])

df[num_cols].describe()

Fixing outliers in numeric columns...


,Age,Number Of Dependants,Income_Lakhs,Annual_Premium_Amount
count,50000.000000,50000.000000,50000.000000,50000.000000
mean,34.455960,1.712080,21.697700,15768.095565
std,13.760307,1.498248,18.925378,8419.772385
min,18.000000,-3.000000,1.000000,3501.000000
25%,22.000000,0.000000,7.000000,8608.000000
50%,31.000000,2.000000,17.000000,13929.000000
75%,45.000000,3.000000,31.000000,22275.250000
max,79.500000,5.000000,67.000000,42776.125000


In [105]:
# Pick target automatically (fallback to last numeric column)
candidate_targets = ['charges','premium','Premium','PremiumAmount','insurance_premium','price','target','label']
target_col = next((c for c in candidate_targets if c in df.columns), None)
if target_col is None:
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    assert len(num_cols) > 0, 'No numeric columns found; add a numeric target column.'
    target_col = num_cols[-1]
target_col

'Annual_Premium_Amount'

In [106]:
# Split features/target and set up preprocessing
# Split features/target
y = df[target_col]
X = df.drop(columns=[target_col])

# Separate numeric and categorical features
num_features = X.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

# Preprocessing: numeric → impute median + scale
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Preprocessing: categorical → impute mode + one-hot encode
try:
    categorical_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
except TypeError:  # fallback for older versions
    categorical_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse=False))
    ])

# Combine preprocessors
preprocess = ColumnTransformer([
    ('num', numeric_transformer, num_features),
    ('cat', categorical_transformer, cat_features)
])


In [107]:
# Choose model (RandomForestRegressor for strong baseline)
model = RandomForestRegressor(n_estimators=400, random_state=42, n_jobs=-1)
pipe = Pipeline([('preprocess', preprocess), ('model', model)])

In [108]:
# Train/test split and fit
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
pipe.fit(X_train, y_train)
preds = pipe.predict(X_test)
mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)
mae, r2

(806.5621700870629, 0.9787351469558159)

In [109]:
# Save artifacts: model + feature schema

joblib.dump(pipe, 'health_premium_model.pkl')

schema = {'target': target_col, 'features': []}

for c in cat_features:
    vals = sorted([str(v) for v in pd.Series(X[c]).dropna().unique().tolist()])
    if len(vals) > 100:
        vals = pd.Series(X[c]).astype(str).value_counts().index.tolist()[:100]
    schema['features'].append({'name': c, 'type': 'categorical', 'values': vals})

for c in num_features:
    s = pd.Series(X[c])
    schema['features'].append({'name': c, 'type': 'numeric', 'min': float(s.min()), 'max': float(s.max()), 'median': float(s.median())})
payload = {'schema': schema, 'metrics': {'MAE': float(mae), 'R2': float(r2)}}

with open('feature_schema.json','w', encoding='utf-8') as f:
    json.dump(payload, f, indent=2)
print('Saved health_premium_model.pkl and feature_schema.json')

Saved health_premium_model.pkl and feature_schema.json
